# 01. Data Understanding & Exploratory Data Analysis

## Student Interest Profile Segmentation Using Clustering

Notebook ini bertujuan untuk memahami struktur, karakteristik, dan kualitas dataset sebelum dilakukan preprocessing dan clustering.

Tahapan yang dilakukan:
1. Dataset overview and structure.
2. Feature identification.
3. Data quality analysis.
4. Profile feature analysis.
5. Interest feature analysis.
6. Correlation analysis.
7. EDA findings and preprocessing recommendations.

Catatan: Notebook ini hanya berfokus pada Data Understanding dan EDA. 
Tidak ada proses data cleaning, scaling, normalization, maupun model development.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

# Mengatur tampilan DataFrame
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Mengatur tampilan visualisasi
sns.set_theme(style="whitegrid")

# Lokasi proyek: folder induk notebooks jika ada folder data di sana (struktur lokal),
# selain itu folder kerja saat ini (Colab atau Kaggle)
PROJECT_ROOT = Path("..") if Path("../data").exists() else Path(".")
DATASET_NAMES = ("Clustering_Marketing.csv", "03_Clustering_Marketing.csv")


def find_dataset():
    # Cek lokasi yang umum dipakai di lokal dan Colab
    for folder in (PROJECT_ROOT / "data", Path("data"), Path("."), Path("/content"), Path("/content/data")):
        for name in DATASET_NAMES:
            if (folder / name).exists():
                return folder / name
    # Di Kaggle, dataset ada di dalam /kaggle/input/<nama-dataset>/
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for name in DATASET_NAMES:
            found = next(kaggle_input.rglob(name), None)
            if found is not None:
                return found
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Taruh Clustering_Marketing.csv di folder data/, "
        "upload ke folder kerja Colab, atau tambahkan sebagai Input di Kaggle."
    )


DATA_PATH = find_dataset()

# Folder untuk menyimpan hasil analisis EDA
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "EDA"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully.")
print(f"Dataset path: {DATA_PATH.resolve()}")

## 2. Load Dataset

Memuat dataset yang digunakan dalam penelitian.

Dataset yang digunakan berasal dari Kaggle:
Students Social Network Profile Clustering.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# Menampilkan lima baris pertama
display(df.head())

## 3. Dataset Structure

Tahap ini digunakan untuk memahami struktur dataset, termasuk:
- Jumlah observasi dan fitur.
- Nama kolom.
- Tipe data setiap kolom.
- Jumlah nilai unik.
- Statistik deskriptif.

In [ ]:
# Informasi umum dataset
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
# Ringkasan karakteristik setiap kolom
column_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique(dropna=False),
    "Missing Values": df.isna().sum()
})

display(column_summary)

column_summary.to_csv(
    TABLE_DIR / "dataset_column_summary.csv"
)

In [ ]:
# Statistik deskriptif seluruh kolom
display(df.describe(include="all").T)

## 4. Feature Identification

Berdasarkan struktur dataset, fitur dibagi menjadi dua kelompok:

1. Profile Features:
   - gradyear
   - gender
   - age
   - NumberOffriends

2. Interest Features:
   - 36 fitur yang merepresentasikan informasi minat pada profil.

Pemisahan ini digunakan untuk mempermudah analisis dataset.
Keputusan final mengenai fitur yang digunakan untuk clustering
akan ditentukan pada tahap preprocessing.

In [ ]:
# Kolom informasi profil
profile_cols = [
    "gradyear",
    "gender",
    "age",
    "NumberOffriends"
]

# Memastikan semua kolom profil tersedia
missing_profile_cols = [
    col for col in profile_cols
    if col not in df.columns
]

if missing_profile_cols:
    raise ValueError(
        f"Profile columns not found: {missing_profile_cols}"
    )

# Mengidentifikasi kolom yang bukan fitur minat
unnamed_cols = [
    col for col in df.columns
    if str(col).startswith("Unnamed:")
]

# Mengidentifikasi fitur minat
interest_cols = [
    col for col in df.columns
    if col not in profile_cols
    and col not in unnamed_cols
]

print("Profile Features:", len(profile_cols))
print(profile_cols)

print("\nInterest Features:", len(interest_cols))
print(interest_cols)

print("\nUnnamed Columns:", unnamed_cols)

if len(interest_cols) != 36:
    print(
        "Warning: Number of interest features "
        "is different from the expected 36."
    )

## 5. Data Quality Analysis

### 5.1 Missing Value Analysis

Mengidentifikasi jumlah dan persentase missing values
pada setiap kolom.

Analisis ini bertujuan mengetahui kualitas data sebelum
menentukan strategi penanganan missing values pada
tahap preprocessing.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

# Hanya mengambil kolom yang memiliki missing values
missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(
    by="Missing Count",
    ascending=False
)

if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary.round(2))

    missing_summary.to_csv(
        TABLE_DIR / "missing_value_summary.csv"
    )

In [ ]:
# Visualisasi missing values
if not missing_summary.empty:

    plt.figure(figsize=(10, 5))

    plt.barh(
        missing_summary.index,
        missing_summary["Missing Percentage"]
    )

    plt.xlabel("Missing Values (%)")
    plt.ylabel("Features")
    plt.title("Missing Value Distribution")

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "missing_value_distribution.png",
        dpi=150
    )

    plt.show()

### 5.2 Duplicate Data Analysis

Memeriksa apakah terdapat baris yang memiliki nilai
identik pada seluruh kolom dataset.

Tujuan analisis ini adalah mengidentifikasi kemungkinan
duplikasi observasi.

Baris duplikat tidak langsung dihapus karena perlu
dipastikan terlebih dahulu apakah duplikasi tersebut
merupakan kesalahan data.

In [ ]:
# Menghitung full duplicate rows
duplicate_count = df.duplicated().sum()

duplicate_percentage = (
    duplicate_count / len(df) * 100
)

print("Total duplicate rows:", duplicate_count)

print(
    f"Duplicate percentage: {duplicate_percentage:.2f}%"
)

# Melihat contoh baris duplikat
duplicate_rows = df[
    df.duplicated(keep=False)
]

display(
    duplicate_rows.head(10)
)

## 6. Profile Feature Analysis

### 6.1 Graduation Year Distribution

Menganalisis distribusi tahun kelulusan untuk mengetahui
jumlah profil pada setiap tahun.

In [ ]:
gradyear_summary = (
    df["gradyear"]
    .fillna("Missing")
    .astype(str)
    .value_counts()
    .sort_index()
)

gradyear_table = pd.DataFrame({
    "Profile Count": gradyear_summary,
    "Percentage": (
        gradyear_summary / len(df) * 100
    )
})

display(gradyear_table)

gradyear_table.to_csv(
    TABLE_DIR / "graduation_year_summary.csv"
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    gradyear_summary.index,
    gradyear_summary.values
)

plt.xlabel("Graduation Year")
plt.ylabel("Number of Profiles")

plt.title("Profile Distribution by Graduation Year")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "graduation_year_distribution.png",
    dpi=150
)

plt.show()

### 6.2 Gender Distribution

Menganalisis distribusi kategori gender untuk mengetahui
proporsi setiap kategori serta jumlah profil yang tidak
memiliki informasi gender.

In [ ]:
gender_summary = (
    df["gender"]
    .fillna("Missing")
    .astype(str)
    .value_counts()
)

gender_table = pd.DataFrame({
    "Profile Count": gender_summary,
    "Percentage": (
        gender_summary / len(df) * 100
    )
})

display(gender_table)

gender_table.to_csv(
    TABLE_DIR / "gender_summary.csv"
)

### 6.3 Age Distribution and Validation

Menganalisis distribusi umur serta memeriksa nilai yang
tidak dapat dikonversi menjadi numerik.

Pemeriksaan ini diperlukan karena kolom age berpotensi
mengandung missing values, format teks yang tidak konsisten,
atau nilai numerik yang tidak wajar.

In [ ]:
# Mengubah kolom age menjadi numerik
# errors="coerce" mengubah nilai yang tidak valid menjadi NaN

age_numeric = pd.to_numeric(
    df["age"],
    errors="coerce"
)

# Memeriksa missing dan invalid values
original_missing_age = df["age"].isnull().sum()

invalid_age = (
    df["age"].notna()
    & age_numeric.isna()
).sum()

print("Original missing age:", original_missing_age)
print("Invalid age format:", invalid_age)

print("\nAge descriptive statistics:")
display(age_numeric.describe())

In [ ]:
# Memeriksa nilai umur yang perlu diinvestigasi
age_outside_range = (
    age_numeric.notna()
    & ~age_numeric.between(13, 25)
)

age_negative = age_numeric < 0
age_above_100 = age_numeric > 100

print(
    "Numeric ages outside 13-25:",
    age_outside_range.sum()
)

print(
    "Negative ages:",
    age_negative.sum()
)

print(
    "Ages above 100:",
    age_above_100.sum()
)

age_check = pd.DataFrame({
    "Original Age": df["age"],
    "Numeric Age": age_numeric
})

display(
    age_check.loc[
        age_outside_range
        | age_negative
        | age_above_100
    ].head(20)
)

In [ ]:
# Melihat contoh nilai age yang tidak valid
invalid_age_values = df.loc[
    df["age"].notna() & age_numeric.isna(),
    "age"
]

print("Jumlah baris age tidak valid:", len(invalid_age_values))
print("Jumlah nilai unik:", invalid_age_values.nunique())

display(
    invalid_age_values.value_counts().head(15)
)

In [ ]:
# Visualisasi seluruh umur numerik
age_valid = age_numeric.dropna()

plt.figure(figsize=(10, 5))

plt.hist(
    age_valid,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Age")
plt.ylabel("Number of Profiles")

plt.title("Age Distribution - All Numeric Values")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "age_distribution_all.png",
    dpi=150
)

plt.show()

In [ ]:
# Visualisasi seluruh umur numerik
age_valid = age_numeric.dropna()

plt.figure(figsize=(10, 5))

plt.hist(
    age_valid,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Age")
plt.ylabel("Number of Profiles")

plt.title("Age Distribution - All Numeric Values")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "age_distribution_all.png",
    dpi=150
)

plt.show()

### 6.4 Number of Friends Distribution

Menganalisis distribusi jumlah teman pada setiap profil.

Pemeriksaan ini mencakup statistik deskriptif, nilai
yang tidak dapat dikonversi, dan kemungkinan nilai ekstrem.

In [ ]:
friends_numeric = pd.to_numeric(
    df["NumberOffriends"],
    errors="coerce"
)

friends_invalid = (
    df["NumberOffriends"].notna()
    & friends_numeric.isna()
).sum()

print(
    "Original missing values:",
    df["NumberOffriends"].isnull().sum()
)

print(
    "Invalid numeric values:",
    friends_invalid
)

print(
    "Negative values:",
    (friends_numeric < 0).sum()
)

print("\nNumber of Friends Statistics:")

display(
    friends_numeric.describe()
)

## 7. Interest Feature Analysis

Bagian ini berfokus pada karakteristik 36 fitur minat.

Fitur-fitur tersebut akan diperiksa untuk mengetahui:
- Statistik deskriptif.
- Validitas nilai.
- Distribusi nilai nol.
- Sparsity.
- Frekuensi kemunculan topik.
- Distribusi dan nilai ekstrem.

In [ ]:
# Membuat salinan fitur minat untuk kebutuhan EDA
X_interest = df[interest_cols].copy()

# Memastikan seluruh fitur dapat diperiksa sebagai numerik
X_interest = X_interest.apply(
    pd.to_numeric,
    errors="coerce"
)

print("Interest feature shape:", X_interest.shape)

print(
    "Missing values after numeric conversion:",
    X_interest.isna().sum().sum()
)

### 7.1 Descriptive Statistics

In [ ]:
interest_statistics = X_interest.describe().T

display(interest_statistics)

interest_statistics.to_csv(
    TABLE_DIR / "interest_descriptive_statistics.csv"
)

### 7.2 Percentile and IQR Analysis

In [ ]:
# Memeriksa distribusi percentile setiap fitur minat
percentile_summary = pd.DataFrame({
    "P95": X_interest.quantile(0.95),
    "P99": X_interest.quantile(0.99),
    "P99.5": X_interest.quantile(0.995),
    "Maximum": X_interest.max()
})

percentile_summary = percentile_summary.sort_values(
    by="P99",
    ascending=False
)

display(percentile_summary.round(2))

# Menghitung berapa fitur yang memiliki P99 <= 2
features_p99_below_2 = (
    percentile_summary["P99"] <= 2
).sum()

print(
    f"Features with P99 <= 2: "
    f"{features_p99_below_2}/{len(interest_cols)}"
)

print(
    "\nFeatures with P99 > 2:"
)

display(
    percentile_summary[
        percentile_summary["P99"] > 2
    ]
)

In [ ]:
# Menghitung Q1, Q3, dan IQR setiap fitur
q1 = X_interest.quantile(0.25)
q3 = X_interest.quantile(0.75)

iqr = q3 - q1

# Membuat ringkasan IQR
iqr_summary = pd.DataFrame({
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr
})

display(iqr_summary)

# Menghitung jumlah fitur berdasarkan nilai IQR
zero_iqr_count = (iqr == 0).sum()
nonzero_iqr_count = (iqr > 0).sum()

print("Features with IQR = 0:", zero_iqr_count)
print("Features with IQR > 0:", nonzero_iqr_count)

print("\nFeatures with non-zero IQR:")
display(iqr[iqr > 0])

### 7.3 Feature Data Validation

Memeriksa apakah fitur minat memiliki nilai:
- Missing values.
- Negative values.
- Non-integer values.

Pemeriksaan ini berguna untuk memahami kualitas data
sebelum menentukan metode preprocessing.

In [ ]:
missing_features = X_interest.isnull().sum()

negative_features = (
    X_interest < 0
).sum()

non_integer_features = (
    X_interest.notna()
    & (X_interest % 1 != 0)
).sum()

validation_summary = pd.DataFrame({
    "Missing Values": missing_features,
    "Negative Values": negative_features,
    "Non-Integer Values": non_integer_features,
    "Data Type": X_interest.dtypes.astype(str)
})

display(validation_summary)

print("\nTotal missing:", X_interest.isna().sum().sum())

print(
    "Total negative:",
    (X_interest < 0).sum().sum()
)

print(
    "Total non-integer:",
    (
        X_interest.notna()
        & (X_interest % 1 != 0)
    ).sum().sum()
)

validation_summary.to_csv(
    TABLE_DIR / "interest_feature_validation.csv"
)

### 7.4 Sparsity Analysis

Sparsity menunjukkan seberapa besar proporsi nilai nol
dalam matriks fitur minat.

Analisis ini penting karena banyaknya nilai nol dapat
memengaruhi perhitungan jarak antarprofil pada algoritma
clustering.

In [ ]:
total_cells = X_interest.size

valid_cells = X_interest.notna().sum().sum()

zero_count = X_interest.eq(0).sum().sum()

missing_count = X_interest.isna().sum().sum()

sparsity = (
    zero_count / valid_cells * 100
    if valid_cells > 0 else np.nan
)

print("Total cells:", total_cells)

print("Valid cells:", valid_cells)

print("Zero values:", zero_count)

print("Missing values:", missing_count)

print(f"Sparsity: {sparsity:.2f}%")

### 7.5 Number of Recorded Interests per Profile

Menghitung jumlah fitur minat yang memiliki nilai lebih
dari nol pada setiap profil.

Analisis ini digunakan untuk mengetahui apakah sebagian
besar profil hanya memiliki sedikit informasi minat.

In [ ]:
topics_per_profile = (
    X_interest.gt(0).sum(axis=1)
)

print("Topics per profile statistics:")

display(
    topics_per_profile.describe()
)

print(
    "\nProfiles with no recorded interest:",
    (topics_per_profile == 0).sum()
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    topics_per_profile,
    bins=np.arange(
        -0.5,
        len(interest_cols) + 1.5,
        1
    ),
    edgecolor="black"
)

plt.xlabel("Number of Positive Interest Features")
plt.ylabel("Number of Profiles")

plt.title("Number of Recorded Interests per Profile")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "topics_per_profile_distribution.png",
    dpi=150
)

plt.show()

### 7.6 All-Zero Interest Profiles

Mengidentifikasi profil yang seluruh 36 fitur minatnya
memiliki nilai nol.

Profil seperti ini perlu diperhatikan karena tidak memiliki
informasi positif pada fitur minat yang digunakan dalam
penelitian.

Pada tahap EDA, profil ini hanya diidentifikasi dan
dilaporkan. Keputusan untuk mempertahankan atau
mengecualikannya akan dilakukan pada tahap preprocessing.

In [ ]:
all_zero_profiles = X_interest.eq(0).all(axis=1)

total_zero_profiles = all_zero_profiles.sum()

zero_profile_percentage = (
    total_zero_profiles / len(X_interest) * 100
)

print(
    "Profiles with all-zero interests:",
    total_zero_profiles
)

print(
    f"Percentage: {zero_profile_percentage:.2f}%"
)

# Melihat contoh profil all-zero
display(
    df.loc[all_zero_profiles, profile_cols].head(10)
)

### 7.7 Topic Distribution

Mengidentifikasi fitur minat yang paling sering muncul
di antara seluruh profil.

Profile Count dihitung berdasarkan jumlah profil dengan
nilai fitur lebih dari nol.

Total Values menunjukkan jumlah seluruh nilai dari
fitur tersebut.

In [ ]:
topic_summary = pd.DataFrame({
    "Profile Count": X_interest.gt(0).sum(),
    "Profile Percentage": X_interest.gt(0).mean() * 100,
    "Total Values": X_interest.sum(),
    "Mean Value": X_interest.mean(),
    "Median Value": X_interest.median(),
    "Maximum Value": X_interest.max()
})

topic_summary = topic_summary.sort_values(
    by="Profile Count",
    ascending=False
)

display(topic_summary)

topic_summary.to_csv(
    TABLE_DIR / "topic_summary.csv"
)

In [ ]:
top_10_topics = (
    topic_summary.head(10)
    .sort_values("Profile Count")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_topics.index,
    top_10_topics["Profile Count"]
)

plt.xlabel("Number of Profiles")
plt.ylabel("Interest Features")

plt.title("Top 10 Most Common Interest Features")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "top_10_interest_features.png",
    dpi=150
)

plt.show()

### 7.8 Feature Distribution and Extreme Values

Menganalisis distribusi setiap fitur minat menggunakan:
- Mean.
- Median.
- Maximum.
- Skewness.
- Jumlah profil dengan nilai positif.

Pemeriksaan ini membantu mengetahui fitur yang memiliki
distribusi tidak simetris atau nilai ekstrem.

Nilai yang teridentifikasi sebagai ekstrem tidak otomatis
dianggap sebagai kesalahan data.

In [ ]:
feature_distribution = pd.DataFrame({
    "Mean": X_interest.mean(),
    "Median": X_interest.median(),
    "Maximum": X_interest.max(),
    "Skewness": X_interest.skew(),
    "Non-Zero Profiles": X_interest.gt(0).sum(),
    "Zero Percentage": X_interest.eq(0).mean() * 100
})

feature_distribution = feature_distribution.sort_values(
    by="Skewness",
    ascending=False
)

display(feature_distribution)

feature_distribution.to_csv(
    TABLE_DIR / "feature_distribution_analysis.csv"
)

In [ ]:
# Memeriksa lima fitur minat dengan frekuensi kemunculan tertinggi
features_to_check = topic_summary.head(5).index.tolist()

for col in features_to_check:

    print(f"\nTop 5 highest values: {col}")

    top_values = X_interest[col].nlargest(5)

    display(
        df.loc[
            top_values.index,
            profile_cols + [col]
        ].sort_values(
            by=col,
            ascending=False
        )
    )

## 8. Correlation Analysis

Analisis korelasi digunakan untuk memahami hubungan
linear antara 36 fitur minat.

Metode Pearson digunakan untuk menghitung korelasi.

Karena dataset memiliki banyak nilai nol, hasil korelasi
perlu diinterpretasikan secara hati-hati dan tidak boleh
langsung dianggap sebagai hubungan sebab-akibat.

In [ ]:
# Menghitung korelasi antar 36 fitur minat
correlation_matrix = X_interest.corr(
    method="pearson"
)

# Menyimpan correlation matrix tanpa menampilkannya
correlation_matrix.to_csv(TABLE_DIR / "correlation_matrix.csv")

In [ ]:
# Visualisasi correlation heatmap

plt.figure(figsize=(16, 13))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0
)

plt.title("Correlation Heatmap of Interest Features")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "interest_correlation_heatmap.png",
    dpi=150
)

plt.show()

In [ ]:
# Menghindari penghitungan pasangan korelasi dua kali
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

# Mengubah matriks menjadi daftar pasangan fitur
correlation_pairs = (
    upper_triangle
    .stack()
)

# Memisahkan korelasi positif dan negatif
positive_correlations = (
    correlation_pairs[correlation_pairs > 0]
    .sort_values(ascending=False)
)

negative_correlations = (
    correlation_pairs[correlation_pairs < 0]
    .sort_values(ascending=True)
)

print("Top 15 Positive Correlations:")
display(
    positive_correlations.head(15).to_frame("Correlation")
)

print("\nTop 15 Negative Correlations:")
display(
    negative_correlations.head(15).to_frame("Correlation")
)

correlation_pairs.to_csv(
    TABLE_DIR / "correlation_pairs.csv"
)

In [ ]:
# Perbandingan korelasi: count-based vs binary (presence/absence)
X_binary = X_interest.gt(0).astype(int)

binary_correlation = X_binary.corr()

binary_upper = binary_correlation.where(
    np.triu(np.ones(binary_correlation.shape), k=1).astype(bool)
)

binary_pairs = binary_upper.stack().sort_values(ascending=False)

top_count_pairs = positive_correlations.head(15)

comparison = pd.DataFrame({
    "Count-Based": top_count_pairs,
    "Binary-Based": binary_pairs.loc[top_count_pairs.index]
})

display(comparison.round(3))

comparison.to_csv(TABLE_DIR / "correlation_count_vs_binary.csv")

In [ ]:
# Visualisasi 1: Prevalensi 36 topik minat
topic_plot_data = (
    topic_summary
    .reset_index()
    .rename(columns={"index": "Topic"})
    .sort_values("Profile Percentage", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 12))

sns.barplot(
    data=topic_plot_data,
    x="Profile Percentage",
    y="Topic",
    color="steelblue",
    ax=ax
)

ax.set_title("Prevalensi Topik Minat (Persentase Profil)")
ax.set_xlabel("Persentase Profil (%)")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_topic_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Visualisasi 2: Heatmap korelasi antar fitur minat
fig, ax = plt.subplots(figsize=(14, 11))

sns.heatmap(
    correlation_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-0.6,
    vmax=0.6,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Korelasi Pearson"},
    ax=ax
)

ax.set_title("Heatmap Korelasi 36 Fitur Minat")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Visualisasi 3: Distribusi jumlah topik per profil
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    topics_per_profile,
    binwidth=1,
    color="steelblue",
    ax=ax
)

ax.axvline(
    topics_per_profile.mean(),
    color="red",
    linestyle="--",
    label=f"Mean = {topics_per_profile.mean():.2f}"
)
ax.axvline(
    topics_per_profile.median(),
    color="orange",
    linestyle="--",
    label=f"Median = {topics_per_profile.median():.0f}"
)

ax.set_title("Distribusi Jumlah Topik per Profil")
ax.set_xlabel("Jumlah Topik yang Disebut (nilai > 0)")
ax.set_ylabel("Jumlah Profil")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_topics_per_profile.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Visualisasi 4: Distribusi NumberOffriends
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    friends_numeric,
    bins=60,
    color="steelblue",
    ax=axes[0]
)
axes[0].set_title("Distribusi NumberOffriends")
axes[0].axvline(
    friends_numeric.median(),
    color="orange",
    linestyle="--",
    label=f"Median = {friends_numeric.median():.0f}"
)
axes[0].legend()

friends_log = np.log10(friends_numeric + 1)

sns.histplot(
    friends_log,
    bins=40,
    color="steelblue",
    ax=axes[1]
)
axes[1].set_title("Distribusi NumberOffriends (Skala Log)")
axes[1].set_xlabel("log10(NumberOffriends + 1)")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_friends_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Duplicate Interest Patterns

Mengidentifikasi profil yang memiliki pola nilai identik
pada seluruh fitur minat.

Pola minat yang sama tidak selalu berarti observasi
berasal dari siswa yang sama.

Oleh karena itu, hasil ini hanya digunakan sebagai
informasi tambahan dalam memahami karakteristik dataset.

In [ ]:
duplicate_interest_count = X_interest.duplicated().sum()

duplicate_interest_percentage = (
    duplicate_interest_count / len(X_interest) * 100
)

print(
    "Duplicate Interest Patterns:",
    duplicate_interest_count
)

print(
    f"Percentage: {duplicate_interest_percentage:.2f}%"
)

# Melihat pola minat yang paling sering muncul
interest_pattern_frequency = (
    X_interest.value_counts()
    .head(10)
    .reset_index(name="Frequency")
)

display(interest_pattern_frequency)

## 10. EDA Findings and Conclusion

Bagian ini merangkum hasil utama dari Data Understanding
dan Exploratory Data Analysis.

Temuan EDA digunakan sebagai dasar pertimbangan dalam
menentukan strategi preprocessing dan feature engineering.

Perlu diingat bahwa temuan statistik tidak secara otomatis
menentukan tindakan pembersihan data. Setiap keputusan
perlu dipertimbangkan dan diuji pada tahap berikutnya.

In [ ]:
findings = []

# 1. Missing values
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0]

missing_evidence = (
    "; ".join(
        f"{col}: {count} ({count / len(df) * 100:.2f}%)"
        for col, count in missing_cols.items()
    )
    if len(missing_cols)
    else "No missing values found."
)

findings.append({
    "Finding": "Missing Values",
    "Evidence": missing_evidence,
    "Implication": (
        "Tentukan strategi penanganan missing values pada tahap preprocessing."
    )
})

# 2. Duplicate rows
findings.append({
    "Finding": "Full Duplicate Rows",
    "Evidence": (
        f"{duplicate_count} rows "
        f"({duplicate_percentage:.2f}%)."
    ),
    "Implication": (
        "Lakukan investigasi sebelum memutuskan apakah baris duplikat perlu dihapus."
    )
})

# 3. Age validation
findings.append({
    "Finding": "Age Data Quality",
    "Evidence": (
        f"{invalid_age} non-missing values cannot be converted to numeric."
    ),
    "Implication": (
        "Perlu investigasi terhadap format age sebelum digunakan dalam analisis lanjutan."
    )
})

# 4. Sparsity
findings.append({
    "Finding": "Interest Feature Sparsity",
    "Evidence": f"{sparsity:.2f}% of valid interest cells are zero.",
    "Implication": (
        "Pertimbangkan karakteristik sparse data dalam pemilihan representasi fitur."
    )
})

# 5. All-zero profiles
findings.append({
    "Finding": "All-Zero Interest Profiles",
    "Evidence": (
        f"{total_zero_profiles} profiles "
        f"({zero_profile_percentage:.2f}%)."
    ),
    "Implication": (
        "Evaluasi apakah profil tanpa informasi minat perlu dipertahankan "
        "atau dipisahkan dalam eksperimen."
    )
})

# 6. Duplicate interest patterns
findings.append({
    "Finding": "Duplicate Interest Patterns",
    "Evidence": (
        f"{duplicate_interest_count} rows "
        f"({duplicate_interest_percentage:.2f}%)."
    ),
    "Implication": (
        "Pola minat identik tidak otomatis merupakan duplikasi identitas siswa."
    )
})

# 7. Feature distribution
skewed_features = feature_distribution[
    feature_distribution["Skewness"].abs() > 2
]

findings.append({
    "Finding": "Skewed Interest Features",
    "Evidence": (
        f"{len(skewed_features)} features have absolute skewness greater than 2."
    ),
    "Implication": (
        "Pertimbangkan transformasi fitur sebagai salah satu eksperimen preprocessing."
    )
})

findings_df = pd.DataFrame(findings)

display(findings_df)

findings_df.to_csv(
    TABLE_DIR / "eda_findings.csv",
    index=False
)

In [ ]:
print("Dataset Overview")
print("----------------")

print("Total Profiles:", len(df))
print("Total Columns:", len(df.columns))
print("Profile Features:", len(profile_cols))
print("Interest Features:", len(interest_cols))

print("\nData Quality")
print("------------")

print("Missing Values:", df.isna().sum().sum())
print("Duplicate Rows:", duplicate_count)

print("\nInterest Characteristics")
print("------------------------")

print(f"Sparsity: {sparsity:.2f}%")
print("All-Zero Profiles:", total_zero_profiles)
print("Duplicate Interest Patterns:", duplicate_interest_count)

print("\nTop 5 Most Common Interests")
print("---------------------------")

display(
    topic_summary[
        ["Profile Count", "Profile Percentage"]
    ].head(5)
)

# Exploratory Data Analysis (EDA)

## 1. Dataset Structure

Dataset terdiri dari **15.000 profil siswa** dengan total **40 kolom**, yang terbagi menjadi:

- **4 fitur profil:** `gradyear`, `gender`, `age`, dan `NumberOffriends`.
- **36 fitur minat:** berupa jumlah kemunculan kata kunci yang merepresentasikan minat siswa.

Secara semantik, 36 fitur minat dapat dikelompokkan menjadi enam kategori:

| Kategori | Jumlah Fitur |
|---|---:|
| Olahraga | 10 |
| Penampilan / Romantis | 8 |
| Musik | 5 |
| Belanja / Merek | 5 |
| Religi | 4 |
| Morbid / Berisiko | 4 |
| **Total** | **36** |

**Distribusi fitur profil:**

- `gradyear`: Tahun kelulusan 2006–2009 relatif seimbang, dengan proporsi sekitar 25% untuk setiap angkatan.
- `gender`: F sebesar 73,7%, M sebesar 17,4%, dan missing values sebesar 8,9%. Dari data gender yang tersedia, sekitar 80,9% merupakan perempuan.
- Seluruh 36 fitur minat memiliki tipe data integer non-negatif dan tidak mengandung missing values.

## 2. Data Quality Assessment

Missing values hanya ditemukan pada fitur profil, yaitu:

| Fitur | Missing Values | Persentase |
|---|---:|---:|
| `age` | 2.496 | 16,64% |
| `gender` | 1.337 | 8,91% |

Selain itu, ditemukan **266 full duplicate rows (1,77%)**. Baris-baris tersebut memiliki seluruh nilai kolom yang identik. Namun, karena dataset tidak menyediakan unique student ID, kesamaan seluruh nilai belum cukup untuk memastikan bahwa baris tersebut merupakan duplikasi identitas siswa.

Kolom `age` merupakan fitur dengan permasalahan kualitas data paling signifikan:

- Terdapat 276 nilai non-missing yang menggunakan format tanggal lokal Jerman, seperti `"16. Jun"`, `"17. Mrz"`, dan `"16. Dez"`, tanpa informasi tahun sehingga tidak dapat dipulihkan sebagai nilai usia.
- Setelah dikonversi menjadi numerik, terdapat 12.228 nilai usia.
- Dari nilai numerik tersebut, ditemukan 153 nilai yang berada di luar rentang usia wajar 13–25 tahun.
- Sebanyak 77 nilai usia berada di atas 100 tahun, dengan nilai maksimum 106,927.
- Dengan demikian, sebanyak 12.075 baris atau sekitar **80,5% dari seluruh profil** memiliki nilai usia numerik dalam rentang 13–25 tahun.

Sebaliknya, seluruh 36 fitur minat memiliki kualitas data yang sangat baik:

- 0 missing values.
- 0 nilai negatif.
- Seluruh nilai bertipe integer.

## 3. Interest Features Analysis

Seluruh fitur minat memiliki **median 0 dan mean di bawah 1**, yang menunjukkan bahwa sebagian besar siswa memiliki sedikit atau bahkan tidak memiliki kemunculan pada setiap kata kunci.

Berdasarkan frekuensi profil yang memiliki nilai positif (Gambar 1), lima topik paling populer adalah:

| Topik | Persentase Profil |
|---|---:|
| Music | 44,4% |
| God | 26,5% |
| Shopping | 25,7% |
| Hair | 22,7% |
| Dance | 22,2% |

Sementara itu, topik dengan frekuensi terendah adalah `bible` (1,5%) dan `marching` (2,9%).

**Interpretasi distribusi fitur minat:**

- Seluruh 36 fitur menunjukkan distribusi *highly right-skewed* dengan nilai skewness di atas 2.
- Fitur `blonde` memiliki skewness tertinggi, yaitu sekitar 117,251, akibat adanya nilai ekstrem sebesar 327.
- Distribusi jumlah topik per profil (Gambar 3) menunjukkan konsentrasi tertinggi di sekitar 3–4 topik, dengan ekor distribusi hingga 31 topik.

### 3.1. Percentile Analysis

Analisis percentile dilakukan untuk memahami distribusi nilai pada bagian atas setiap fitur minat serta mengidentifikasi potensi nilai ekstrem.

Hasil pemeriksaan percentile menunjukkan:

- Sebanyak 17 dari 36 fitur memiliki P99 ≤ 2 (47,2%).
- Sebanyak 19 dari 36 fitur memiliki P99 > 2 (52,8%).
- Nilai P99 berada dalam rentang 1–5.
- Nilai P99.5 mencapai maksimum 7 pada beberapa fitur.

Beberapa fitur menunjukkan perbedaan yang cukup besar antara nilai P99 dan nilai maksimum:

| Fitur | P99 | Maximum |
|---|---:|---:|
| `blonde` | 2 | 327 |
| `sex` | 4 | 114 |
| `god` | 5 | 52 |
| `swimming` | 3 | 31 |
| `dance` | 5 | 29 |

Temuan ini menunjukkan adanya ekor distribusi yang panjang (*heavy right tail*) dan nilai ekstrem pada beberapa fitur.

Oleh karena itu, transformasi `log1p` dipertimbangkan untuk mengurangi skewness dan pengaruh nilai ekstrem tanpa langsung menghilangkan informasi dari observasi tersebut.

## 4. Data Sparsity Analysis

Dataset memiliki tingkat sparsity yang cukup tinggi:

- **87,80%** dari seluruh sel fitur minat bernilai nol.
- Rata-rata setiap profil hanya memiliki sekitar **4,39 topik** yang tercatat.
- Sebanyak **1.238 profil (8,25%)** tidak memiliki minat yang tercatat sama sekali (*all-zero interest profiles*).
- Sebanyak **4.163 baris (27,75%)** merupakan pengulangan pola nilai pada 36 fitur minat yang telah muncul sebelumnya.

Profil all-zero tidak selalu berarti siswa tersebut benar-benar tidak memiliki minat. Hal ini hanya menunjukkan bahwa tidak terdapat nilai positif pada 36 kategori minat yang tersedia di dataset.

Temuan tambahan dari Gambar 4 menunjukkan bahwa distribusi `NumberOffriends` juga sangat *right-skewed*:

- Median jumlah teman adalah 20.
- Nilai maksimum mencapai 605.
- Sekitar 18% profil memiliki 0 teman.

Hal ini mengindikasikan adanya subpopulasi profil dengan informasi sosial yang sangat terbatas atau kemungkinan tingkat aktivitas yang rendah.

## 5. Feature Relationships Analysis

Berdasarkan matriks korelasi Pearson (Gambar 2), hubungan positif terkuat ditemukan pada pasangan fitur yang secara semantik memiliki keterkaitan:

| Pasangan Fitur | Korelasi Pearson |
|---|---:|
| band – marching | 0,53 |
| hollister – abercrombie | 0,46 |
| god – jesus | 0,42 |
| jesus – bible | 0,35 |
| kissed – hair | 0,34 |

Tidak ditemukan korelasi negatif yang bermakna. Nilai korelasi negatif yang ditemukan memiliki magnitudo sangat kecil, dengan nilai terendah sekitar -0,018.

**Perbandingan korelasi count vs. binary** menunjukkan beberapa pola menarik:

1. **Topik religi:** Korelasi `god–jesus` menurun dari 0,42 menjadi 0,17 pada representasi binary. Hal ini mengindikasikan bahwa hubungan pada data count sebagian dipengaruhi oleh intensitas penyebutan.
2. **Topik romantis:** Korelasi `sex–kissed` sedikit meningkat dari 0,28 menjadi 0,32 pada representasi binary.
3. **Topik musik dan merek:** Pasangan `band–marching` (0,53 menjadi 0,38) dan `hollister–abercrombie` (0,46 menjadi 0,47) menunjukkan hubungan yang relatif konsisten pada kedua representasi.

**Keterbatasan analisis korelasi:**

- Korelasi pada data count dapat mencampurkan hubungan antar-topik dengan faktor intensitas aktivitas profil.
- Pearson correlation pada data yang sangat sparse berpotensi memberikan gambaran hubungan yang kurang representatif terhadap pola asosiasi sebenarnya.
- Korelasi tidak menunjukkan hubungan sebab-akibat (*correlation does not imply causation*).

## 6. Preprocessing Recommendations

Berdasarkan hasil EDA dan evaluasi kualitas data, berikut rekomendasi preprocessing sebelum proses clustering:

| No. | Fitur / Proses | Rekomendasi |
|---|---|---|
| 1 | `age` | Nilai berformat tanggal Jerman dan usia di luar rentang 13–25 diubah menjadi `NaN`. Gunakan hanya untuk profiling cluster, bukan sebagai fitur clustering. |
| 2 | `gender` | Missing values diubah menjadi kategori `Unknown`. Gunakan hanya untuk profiling cluster. |
| 3 | Full Duplicate Rows | Lakukan investigasi sebelum penghapusan. Dokumentasikan jumlahnya dan pertimbangkan perbandingan hasil dengan dan tanpa full duplicates karena tidak tersedia unique student ID. |
| 4 | All-zero Profiles | Pisahkan 1.238 profil tanpa informasi minat sebagai segmen tersendiri (*no-interest-recorded segment*), kemudian lakukan clustering pada profil lainnya. Jumlah ini perlu dihitung ulang setelah kebijakan duplicate ditentukan. |
| 5 | Feature Representation | Uji dua representasi fitur: `log1p + standardization` untuk mempertahankan informasi intensitas dan binary presence/absence untuk mengisolasi pola keberadaan topik. |
| 6 | Extreme Values | Gunakan transformasi `log1p` untuk mengurangi skewness dan pengaruh nilai ekstrem. Winsorization tidak diterapkan secara otomatis. Dampak clipping berbasis percentile dapat dievaluasi sebagai eksperimen alternatif apabila diperlukan. |
| 7 | Feature Selection | Gunakan hanya 36 fitur minat sebagai input clustering. Fitur profil digunakan untuk interpretasi dan karakterisasi cluster. |
| 8 | Outlier Removal | Tidak dilakukan penghapusan outlier berbasis IQR secara otomatis. Sebanyak 33 fitur memiliki IQR = 0, sedangkan 3 fitur (`music`, `god`, dan `shopping`) memiliki IQR = 1. Pendekatan IQR berpotensi mengidentifikasi nilai positif sebagai outlier, padahal nilai tersebut dapat mengandung informasi penting. |

## 7. EDA Conclusion

Berdasarkan hasil Exploratory Data Analysis, dataset memiliki kualitas fitur minat yang baik, tetapi menunjukkan karakteristik sparsity tinggi, distribusi yang sangat tidak seimbang, serta beberapa permasalahan kualitas data pada fitur profil.

Beberapa temuan utama adalah:

- Terdapat 36 fitur minat tanpa missing values, nilai negatif, maupun nilai non-integer.
- Sebanyak 87,80% sel fitur minat bernilai nol.
- Seluruh fitur menunjukkan distribusi right-skewed, sehingga transformasi data perlu dipertimbangkan.
- Terdapat 1.238 profil tanpa informasi minat yang tercatat.
- Beberapa pasangan fitur menunjukkan korelasi positif yang selaras dengan keterkaitan topiknya.
- Analisis P99 dan IQR menunjukkan adanya nilai ekstrem serta distribusi yang sangat sparse.

Berdasarkan temuan tersebut, proses clustering difokuskan pada **36 fitur minat**, dengan membandingkan dua pendekatan representasi fitur, yaitu:

1. **Count-based representation:** menggunakan transformasi `log1p` dan standardisasi untuk mempertahankan informasi intensitas kemunculan setiap topik.
2. **Binary-based representation:** menggunakan nilai presence/absence untuk mengidentifikasi pola keberadaan minat tanpa mempertimbangkan intensitas kemunculannya.

Sementara itu, fitur profil seperti `gradyear`, `gender`, `age`, dan `NumberOffriends` digunakan untuk membantu interpretasi dan karakterisasi setiap cluster, bukan sebagai input utama clustering.

Hasil EDA ini menjadi dasar dalam menentukan strategi preprocessing dan eksperimen model clustering pada tahap selanjutnya.